In [ ]:
#| default_exp game/piece

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np

??show

#| export
import logging

for logger_name in ("uvicorn", "uvicorn.error", "uvicorn.access"):
    uv_logger = logging.getLogger(logger_name)
    file_handler = logging.FileHandler('HexServer.txt')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    uv_logger.addHandler(file_handler)


show(Strong("I am strong text"))

sakura_headers = [Link(href='https://cdn.jsdelivr.net/npm/sakura.css/css/sakura.css', rel='stylesheet', type='text/css')]



app = FastHTML(hdrs = sakura_headers) # A FastHTML app, including the sakura CSS link in the headers
rt = app.route
server = JupyUvi(app) # Starts a server on port 8000 hosting the app
     


In [ ]:
#| export
from monsterui.all import *


In [ ]:
#| export
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging

In [ ]:
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame

In [ ]:
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap
from HexMagic.core import Terrain, DrainageBasins

In [ ]:
from HexMagic.styles import StyleCSS, SVGBuilder

In [ ]:
app, rt, hexGameServer  = appRoutes()

In [ ]:
!tail -10 base.text

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder

from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion , windy_edge , unique_windy_edge ,HexLegend
from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain
Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins, Watershed

In [ ]:
#| export
from HexMagic.game.data import ActiveGame, GameStorage, TerrainTemplate, Settlement, Piece, CountryFlag

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User

## Helpers

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

In [ ]:
!cat ../../HexMagic/game/data.py

In [ ]:
!cat ../../docs/ll*

## Database

#| export
# Helper to ensure we have a user row
def ensure_user(session) -> int:
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    uid = session['userid']
    row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
    if not row:
        from datetime import datetime
        now = int(datetime.now().timestamp())
        globalStore.users.insert({
            'id': uid, 'username': f'player_{uid}', 'email': '', 'password': '',
            'created': now, 'sessionID': str(uid), 'activeWorld': 0
        })
    return uid



def new_game_page():
    templates = TerrainTemplate.maps  # {'bayArea': 'bayArea_map', ...}
    form = Form(
        Div(
            Label("Map Template", cls="label"),
            Select(
                *[Option(name, value=name) for name in sorted(templates.keys())],
                name="template_name", cls="select select-bordered w-full"
            ),
            cls="form-control"
        ),
        Div(
            Label("Kingdoms", cls="label"),
            Input(type="number", name="kingdoms", value="5", min="1", max="10",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Lakes", cls="label"),
            Input(type="number", name="lakes", value="1", min="0", max="5",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Hex Radius", cls="label"),
            Input(type="number", name="radius", value="25", min="10", max="40",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Button("Create World", type="submit", cls="btn btn-primary mt-4"),
        action="/create_world", method="post",
        cls="card bg-base-200 shadow-lg p-6 space-y-4 max-w-md mx-auto"
    )
    return Titled("New Game",
        Div(
            H3("Choose Your World", cls="text-2xl font-bold text-center mb-6"),
            form,
            cls="flex flex-col items-center p-12"
        )
    )




#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


_game_cache = {}
_cache_lock = threading.Lock()

@patch
def active_board(self: GameStorage, user_id) -> ActiveGame:
    with _cache_lock:
        if user_id in _game_cache:
            logging.info(f"active_board: cache hit for user {user_id}")
            return _game_cache[user_id]
    
    logging.info(f"active_board: cache miss, building for user {user_id}")
    row = self.db.execute("SELECT activeWorld FROM user WHERE id = ?", [user_id]).fetchone()
    if not row or not row[0]:
        return None
    
    world_id = row[0]
    try:
        # gameboard() now loads cover internally and reuses cover.basin
        board = self.gameboard(world_id)
        active = ActiveGame(board=board, cover=board.cover, world_id=world_id)
        
        with _cache_lock:
            _game_cache[user_id] = active
        
        return active
    except Exception as e:
        logging.error(f"active_board: FAILED: {e}", exc_info=True)
        return None


## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

_game_cache = {}
_cache_lock = threading.Lock()

def invalidate_cache(user_id):
    """Call this when creating a new game or the board changes."""
    with _cache_lock:
        _game_cache.pop(user_id, None)


In [ ]:
@rt("/piece_detail/{id}")
def piece_detail(session, id: str):
    """Placeholder for piece detail — right panel."""
    return Div(
        H4(f"🧑 Piece", cls="font-bold"),
        P(f"ID: {id[:12]}…", cls=TextPresets.muted_sm),
        Divider(),
        P("Actions and stats coming soon", cls="opacity-50 text-sm"),
        cls="space-y-2"
    )


## `zoom_region_fast` — Deep Dive

This is the **main pipeline** for the question: *"I have a HexRegion on the coarse grid — give me a high-resolution Terrain covering just that region, with weather and watersheds."*

It orchestrates every piece we've built so far into a 10-step pipeline that stays in numpy-land as long as possible, only constructing the expensive `HexGrid` geometry once at the very end.

### The pipeline at a glance

```
HexRegion (coarse)
    │
    ▼
┌─────────────────┐
│ 1. Bounding box  │  O(|region|)
│ 2. Chunk refs    │  O(bbox area / spacing²)
│ 3. Stitch chunks │  O(chunks × chunk_size)  ← numpy block copies
│ 4. Build mapper  │  O(1) closure creation
│ 5. Region filter │  O(merged_size)  ← identifies fine hexes in region
│ 6. Crop proxy    │  O(cropped_size) ← numpy 2D slice
│ 7. Wrap mapper   │  O(1) closure wrapping
│ 8. Build Terrain │  O(cropped_size) ← HexGrid constructed ONCE here
│ 9. Project weather│  O(valid_fine_hexes) ← vectorized numpy scatter
│10. Project sheds │  O(valid_fine_hexes) ← dict grouping + lookup
└─────────────────┘
    │
    ▼
ZoomResult(terrain, basins, mapper, chunks_loaded)
```

### Step 1 — Bounding box with padding

Given a `HexRegion` (a set of coarse hex indices), we find the axis-aligned bounding box in row/col space:

$$r_{\min}, r_{\max}, c_{\min}, c_{\max} = \text{region\_bounding\_box}(\text{region})$$

Then we **pad** by `halo_rings + 1` in every direction (clamped to grid bounds). The padding ensures that when we zoom, fine hexes at the region boundary have enough neighbor data for interpolation and drainage flow:

$$r_{\min}' = \max(0,\; r_{\min} - p) \qquad r_{\max}' = \min(N_r - 1,\; r_{\max} + p)$$

where $p = \text{halo\_rings} + 1$.

**Why halo_rings?** Each chunk already has a halo of extra hexes for seamless stitching. The `+1` ensures we don't get edge artifacts on the region boundary itself.

### Step 2 — Chunk discovery

`bbox_to_chunk_refs` samples the padded bounding box at `spacing = 2 × rings` intervals and maps each sample point to its chunk via `world_to_chunk`:

```
Padded bbox on coarse grid:
┌─────────────────────────┐
│  ·  ·  ·  ·  ·  ·  ·   │   · = sample points at spacing intervals
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   Each · → world_to_chunk → ChunkRef
│                         │
│  ·  ·  ·  ·  ·  ·  ·   │   + corners checked explicitly
└─────────────────────────┘
```

The result is a `set[ChunkRef]` — deduplicated because multiple sample points can land in the same chunk.

**Corner check:** The loop at `spacing` intervals can miss chunks that *just barely* overlap a corner. The explicit corner sampling catches those.

### Step 3 — Stitch chunks (the heavy lift)

`stitch_chunks_fast` does:

1. For each `ChunkRef`, look up the cached zoomed chunk as a `DataProxy`
2. On cache miss, call `load_or_generate_chunk` (runs the actual zoom computation), then reload as `DataProxy`
3. Compute the merged grid dimensions from the range of coarse positions × scale
4. Copy each chunk's 2D elevation/watershed arrays into the merged array at its offset

The key equations for merged dimensions:

$$H_{\text{merged}} = (r_{\max}^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale} + H_{\text{chunk}}$$
$$W_{\text{merged}} = (c_{\max}^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale} + W_{\text{chunk}}$$

And the placement offset for each chunk:

$$\text{row\_off}^{(k)} = (r_k^{\text{coarse}} - r_{\min}^{\text{coarse}}) \times \text{scale}$$
$$\text{col\_off}^{(k)} = (c_k^{\text{coarse}} - c_{\min}^{\text{coarse}}) \times \text{scale}$$

This is all numpy 2D slice assignment — no per-hex loops.

### Step 4 — Build the mapper (on uncropped proxy)

This is the `build_fine_to_coarse_mapper` from the previous deep dive. The critical point: it's built on the **uncropped** merged proxy, because we need it in Step 5 before we know what to crop.

### Step 5 — Identify the region's fine hexes

This is the step that connects the coarse `HexRegion` to the fine grid. For every valid fine hex:

$$\text{coarse\_idx} = \text{mapper}(\text{fine\_idx})$$

If `coarse_idx ∈ region.hexes`, this fine hex "belongs" to the region.

```
Coarse region (shaded):          Fine merged grid:
┌──┬──┬──┬──┬──┐                 ┌──┬──┬──┬──┬──┬──┬──┬──┬──┬──┐
│  │  │░░│░░│  │   mapper        │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│░░│  │   ←────────     │  │  │  │  │▓▓│▓▓│▓▓│▓▓│  │  │
│  │  │░░│  │  │                 │  │  │  │  │▓▓│▓▓│  │  │  │  │
│  │  │  │  │  │                 │  │  │  │  │  │  │  │  │  │  │
└──┴──┴──┴──┴──┘                 └──┴──┴──┴──┴──┴──┴──┴──┴──┴──┘

                                  ▓▓ = fine hexes where mapper → region hex
```

Three types of fine hexes get marked **invalid** here:
- Already in `invalidRegion` (from stitching)
- Elevation ≤ −99.0 (deep water sentinel)
- Mapper returns −1 (outside any chunk)

This is the most expensive loop — `O(merged_size)` — but it's pure Python integer ops, no geometry.

### Step 6 — Crop the proxy

The merged proxy can be much larger than the region. Imagine a region that's 5 coarse hexes in a vertical strip — the merged proxy might be 4 chunks wide but we only need a narrow column.

$$\text{crop\_padding} = \max(4,\; \text{scale} \times (\text{halo\_rings} + 1))$$

The crop finds the bounding box of `region_fine_indices`, pads it, and extracts a numpy 2D slice:

```
Before crop (merged proxy):      After crop:
┌────────────────────────┐        ┌──────────┐
│                        │        │          │
│       ┌──────┐         │   →    │ ┌──────┐ │
│       │region│         │        │ │region│ │
│       └──────┘         │        │ └──────┘ │
│                        │        │          │
└────────────────────────┘        └──────────┘
    50×80 = 4000 cells              20×30 = 600 cells
```

**Bail-out:** If cropping would save less than 10% of cells, we skip it (the overhead isn't worth it).

The function returns the crop offsets `(trim_top, trim_left)` needed to translate cropped coordinates back to merged coordinates.

### Step 7 — Wrap the mapper

Since the proxy was cropped, fine indices shifted. `make_cropped_mapper` wraps the original mapper:

$$\text{merged\_row} = \text{cropped\_row} + \text{trim\_top}$$
$$\text{merged\_col} = \text{cropped\_col} + \text{trim\_left}$$
$$\text{merged\_idx} = \text{merged\_row} \times W_{\text{old}} + \text{merged\_col}$$

Then feeds `merged_idx` to the original uncropped mapper. If no cropping happened (trim = 0), the original mapper is returned as-is.

### Step 8 — Build Terrain ONCE

`proxy_to_terrain(cropped_proxy)` is the **only** place where `HexGrid` geometry gets constructed. This is the payoff of the entire DataProxy design:

- `stitch_chunks_fast` → numpy only
- `crop_proxy_to_region` → numpy only
- Steps 5-7 → integer arithmetic only
- **Here** → create hex objects, compute pixel positions, etc.

We do it on the **cropped** proxy, so the HexGrid is as small as possible.

### Step 9 — Project weather (vectorized)

Weather lives on the coarse terrain. We need it on the fine terrain. The mapper gives us the correspondence, and we vectorize it:

```python
fine_arr   = [  0,   1,   2,   5,   6, ...]   # fine indices (valid)
coarse_arr = [ 42,  42,  42,  43,  43, ...]   # their coarse parents
```

For each weather field (temperature, precipitation, etc.):

$$\text{fine\_field}[\text{fine\_arr}] = \text{coarse\_field}[\text{coarse\_arr}]$$

This is a single numpy **fancy-index scatter** — no Python loop over hexes. Every fine hex in a `scale × scale` block gets the same coarse value (nearest-neighbor projection).

**Fallback:** If temperature or precipitation still isn't populated (e.g. the coarse terrain didn't have them), it calls `merged_terrain.compute_weather()` to generate from scratch.

### Step 10 — Project watersheds

We group the fine→coarse mapping into `fine_regions`:

$$\text{fine\_regions}[c] = \{f \;|\; \text{mapper}(f) = c\}$$

This is exactly the dict that `project_watersheds` expects — it assigns each fine hex the watershed ID of its coarse parent from the `DrainageBasins`.

The coarse `DrainageBasins` is lazily computed once (Step 3's `self.basin` check) and then reused for all subsequent zooms.

---

### Worked example

Setup:
- `cover` with `rings=3`, `halo_rings=1`, coarse grid `20×20`
- Region = 6 coarse hexes forming an L-shape
- `scale = 2`

```
Step 1:  bbox = rows [4,8], cols [6,9]
         padded = rows [2,10], cols [4,11]  (padding = 2)

Step 2:  At spacing=6, samples hit chunks (0,0,0) and (1,0,-1)
         → 2 ChunkRefs

Step 3:  Each chunk zooms to 14×14 fine hexes
         Merged proxy = 14×28 = 392 cells  (chunks side by side)

Step 4:  Mapper closure built (captures chunk_offsets, scale=2, etc.)

Step 5:  Loop over 392 fine cells:
         - 168 map to region hexes → region_fine_indices
         - 180 map to non-region hexes → ignored
         - 44 invalid (water/edge)

Step 6:  region_fine_indices bbox = rows [4,17], cols [4,17]
         Crop to 18×18 = 324 cells (saves 17%)
         trim_top=4, trim_left=4

Step 7:  Mapper wrapped: cropped_idx → +4 to both row,col → original mapper

Step 8:  HexGrid(18, 18, radius=5) built — 324 hexes with geometry

Step 9:  temperature[fine_arr] = coarse_temp[coarse_arr]  (one numpy op per field)

Step 10: fine_regions = {42: {0,1,2,3}, 43: {4,5,6,7}, ...}
         Each set gets its parent's watershed_id
         → ZoomResult with 168 meaningful fine hexes
```

### Performance characteristics

| Step | Cost | Dominates when... |
|------|------|-------------------|
| 3. Stitch | O(chunks × chunk_size) | Many chunks, cache cold |
| 5. Region filter | O(merged_size) | Large merged proxy |
| 8. Build Terrain | O(cropped_size) | Large region |
| 9. Weather | O(valid_fine × fields) | Many weather fields |

The cache-cold case (first zoom) is dominated by Step 3 — actually computing the zoomed terrain. Subsequent zooms hit the cache and Steps 5-9 dominate, which are all numpy-fast.

### The returned ZoomResult

```python
@dataclass
class ZoomResult:
    terrain: Terrain          # Full Terrain with HexGrid, elevations, weather fields
    basins: DrainageBasins    # Watersheds projected from coarse → fine
    mapper: Callable          # cropped_fine_idx → coarse_idx (for further lookups)
    chunks_loaded: int        # How many chunks were involved
```

The `mapper` is returned so callers can do their own fine→coarse lookups (e.g., "which biome does this fine hex inherit?" or "what political territory is it in?"). It's the cropped version, so it works directly with indices into `terrain.hexGrid`.

## Piece View — Project Plan
That's a really clean three-layer design:

1. **Kingdom Directive** — strategic intent ("we're starving", "explore", "expand")
2. **Piece Orders** — the directive gets mapped onto each piece based on its type and base stats
3. **Moves** — resolved outcomes after all kingdoms' orders collide

So the directive acts like the queen bee's pheromone signal — it shifts *what* each piece does, but *how well* they do it still depends on their base stats. A "gather food" directive would assign harvest orders to pawns (who are great at it) but might also reassign knights to patrol the borders instead of scouting.

For the directive enum, something like:

- **FEED** — prioritize harvest
- **EXPLORE** — prioritize scouting / vision
- **EXPAND** — claim new territory
- **DEFEND** — protect capital and settlements
- **ATTACK** — target enemy pieces/settlements

These look right. So we would to combine them to make a patrol concept. we could have a gosper curve that a piece would through as it moves. if we see something and it wants to attack then it could

That's a brilliant use of the Gosper curve — it's space-filling, so a piece on patrol would efficiently cover an area without a lot of backtracking. The concept would be something like:

    Generate patrol path — a Gosper curve anchored to a settlement or border, sized to fit the piece's territory
    Each turn — the piece advances along the curve by move_strength hexes
    At each hex — check sight range for triggers (enemy pieces, unclaimed resources, etc.)
    Interrupt — if something is spotted, the piece breaks off patrol and generates a new order (attack, harvest, scout closer)

So a patrol is really a streaming order — it keeps going until something more interesting happens. The directive level would control where the patrol is anchored and what triggers a break:

    DEFEND → patrol around capital, interrupt on enemy sighting → attack
    EXPLORE → patrol outward from borders, interrupt on discovery → report back
    FEED → patrol farmable land, interrupt on good hex → harvest


Nice. So let's sketch the Order shape. Something like:
Copied!

class OrderType(Enum):
    PATROL = "patrol"
    ATTACK = "attack"
    MOVE   = "move"
    HARVEST = "harvest"
    SCOUT  = "scout"




In [ ]:
??HexDragMap

So we want to start to create a editbable piece detail. I was thinking about a hexTouchMap to determine which way the piece is facing and a HexDragMap so that we can create Gosper curves of instructions


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great combo! A small hex rose for facing and a drag surface for path drawing map nicely onto those two widgets.

Let's break this into two pieces:

1. **Facing selector** — a tiny `HexGrid.centered(rings=1)` with `HexTouchMap`. Center hex = the piece, tap a neighbor = set `facing` (0–5). That's 7 hexes total, very compact for a detail panel.

2. **Patrol path editor** — a larger `HexDragMap` where dragging across hexes records a sequence of direction transitions, which you then encode as an L-system rule string (`F`, `+`, `-`).

The facing selector is the simpler one — want to start there? The key decision is: how should the 6 direction hexes map to `facing` indices? Right now `Piece.facing` is 0–5, and `HexPosition._orderDirs` goes `[SW, W, NW, NE, E, SE]`. Should the tap targets follow that same ordering?

In [ ]:
??PrimitiveDemo.sampleGrid

In [ ]:
fill = "white"
hexStyle = StyleCSS("HexStyle",fill=fill,stroke="blue",stroke_width=2)
labelStyle = StyleCSS("labelStyle",fill=fill,stroke="black",stroke_width=1)

grid = HexGrid.centered(2,radius=40,style=hexStyle)

#Need to add styles as we go along
grid.builder.add_style(labelStyle)

#itterate through the hexes
for i in range(len(grid.hexes)):
    
    #grid.hexes[i].label = str(i) 
    grid.hexes[i].labelStyle = labelStyle.name

grid.update()

#mark the treasure
grid.hexes[grid.midpoint].label = "X"

#lets have a directions layer
arrowLayer = ""

#Use the realtive position class called HexPosition
for position in HexPosition.directions():
    i = grid.hexposition_to_index(position, 12)
    grid.hexes[i].label = position.label
    arrowLayer += grid.arrow(12,i)

grid.update()

#we can add a layer or update one in a builder using the adjust method
grid.builder.adjust("arrows",arrowLayer)

show(HexTouchMap(grid))

Got the directions going

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice! That's a clean compass rose. The natural next step is wiring up `on_click` so tapping a direction hex posts back the facing index.

You've got the mapping already — `HexPosition.directions()` gives `[SW, W, NW, NE, E, SE]` which is indices 0–5, matching `Piece.facing`. So your `on_click` callback just needs to map the grid index back to a direction index. Something like: build a dict from `grid.hexposition_to_index(dir, middle) → facing_idx` for each of the 6 directions, and return the appropriate `hx_post` attrs.

Want to wire that up, or shall I sketch the callback?

Can you finish this. It probably should have its own route

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# === Facing Selector Widget ===

_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]

def facing_grid(piece_id: str, current_facing: int = 0, radius: int = 30):
    """Build a 1-ring compass rose for selecting piece facing."""
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)
    lbl    = StyleCSS("fc-label",  font_size="13px")

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle
    grid.hexes[mid].style = center
    grid.hexes[mid].label = "⬡"
    grid.hexes[mid].labelStyle = lbl.name

    # Build grid-index → facing-index map
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = lbl.name
        if fi == current_facing:
            grid.hexes[gi].style = active

    grid.update()

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block"
    )


@rt("/facing_selector/{piece_id}")
def facing_selector(session, piece_id: str):
    """Render the facing compass for a piece."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    return facing_grid(piece_id, current_facing=piece.facing)


@rt("/set_facing/{piece_id}/{facing:int}")
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing and return refreshed compass."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    piece.facing = facing % 6
    return facing_grid(piece_id, current_facing=piece.facing)
```

The flow is:

- **`/facing_selector/{piece_id}`** — GET renders the compass rose with the piece's current facing highlighted in green
- **Tap a direction hex** → `hx-post` to **`/set_facing/{piece_id}/{facing}`** which updates `piece.facing` and swaps in the refreshed widget (newly selected hex turns green)
- The `target_id` uses the first 8 chars of the piece UUID so multiple selectors on the same page won't collide

You can embed it in your existing `piece_detail` route with:

```python
Div(hx_get=f"/facing_selector/{id}", hx_trigger="load")
```

Or just call `facing_grid(id, piece.facing)` directly if you already have the piece in hand. Want to test it standalone first, or move on to the patrol drag editor?

## Direction Picker

In [ ]:
# === Facing Selector Widget ===

_DIR_ARROWS = ["↙", "←", "↖", "↗", "→", "↘"]

def facing_grid(piece_id: str, current_facing: int = 0, radius: int = 30):
    """Build a 1-ring compass rose for selecting piece facing."""
    base   = StyleCSS("fc-hex",    fill="#f8f8f8", stroke="#999", stroke_width=1.5)
    active = StyleCSS("fc-active", fill="#4CAF50", stroke="#333", stroke_width=2)
    center = StyleCSS("fc-center", fill="#e0e0e0", stroke="#666", stroke_width=1.5)
    lbl    = StyleCSS("fc-label",  font_size="13px")

    grid = HexGrid.centered(1, radius=radius, style=base)
    for s in [active, center, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle
    grid.hexes[mid].style = center
    grid.hexes[mid].label = "⬡"
    grid.hexes[mid].labelStyle = lbl.name

    # Build grid-index → facing-index map
    dir_map = {}
    for fi, d in enumerate(HexPosition.directions()):
        gi = grid.hexposition_to_index(d, mid)
        dir_map[gi] = fi
        grid.hexes[gi].label = _DIR_ARROWS[fi]
        grid.hexes[gi].labelStyle = lbl.name
        if fi == current_facing:
            grid.hexes[gi].style = active

    grid.update()

    target_id = f"facing-{piece_id[:8]}"

    def on_click(g, i):
        if i in dir_map:
            return {
                "hx-post":   f"/set_facing/{piece_id}/{dir_map[i]}",
                "hx-target": f"#{target_id}",
                "hx-swap":   "outerHTML",
            }
        return {}

    return Div(
        H5("Facing", cls="font-bold text-sm mb-1"),
        HexTouchMap(grid, on_click=on_click),
        id=target_id,
        cls="inline-block"
    )


@rt("/facing_selector/{piece_id}")
def facing_selector(session, piece_id: str):
    """Render the facing compass for a piece."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    return facing_grid(piece_id, current_facing=piece.facing)


@rt("/set_facing/{piece_id}/{facing:int}")
def set_facing(session, piece_id: str, facing: int):
    """Update piece facing and return refreshed compass."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")
    piece = next((p for p in active.board.pieces if p.id == piece_id), None)
    if not piece: return P("Piece not found")
    piece.facing = facing % 6
    return facing_grid(piece_id, current_facing=piece.facing)


## Routes, Pieces, and Gosper

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }

In [ ]:
def showPieces():
    users_df = pd.DataFrame(globalStore.pieces())
    print("pieces:")
    print(users_df)

In [ ]:
showPieces()

In [ ]:
def showActiveSettlements(session):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}")
    active = globalStore.active_board(uid)
    ret = []
    for country in active.board.kingdoms:
        ret.extend( country.settlements)
    return ret


In [ ]:
showActiveSettlements(dummySession)

In [ ]:
def listSettlements():
    users_df = pd.DataFrame(globalStore.settlements())
    print("pieces:")
    print(users_df)

In [ ]:
listSettlements()

In [ ]:
dummySettlment = "e03264bc-9fa4-4fc6-9ca7-6e97e6f088e0"

In [ ]:
def showSettlement(session, settlement_id:str):
    uid = ensure_user(session)
    logging.info(f"showSettlement: uid={uid}\tsettlement_id={settlement_id}")
    active = globalStore.active_board(uid)
    if not active:
        logging.info(f"kingdom: not active")
        return RedirectResponse('/', status_code=303)

    try:
        place = globalStore.settlement_from_id(settlement_id)

        #board = active.board
        terrain = active.board.terrain
        region = place.region(terrain.hexGrid)
        logging.info(f"showSettlement[{settlement_id}]: {len(region.hexes) }")
        if len(region.hexes) == 0:
            return None
        # Ensure cover is wired to storage for chunk caching
        #cover.db = self

        # zoom_region reuses cover.basin — no O(n²) recompute
        return active.cover.zoom_region_fast(region,  compute_weather=True)
    except Exception as e:
        logging.error(f"showSettlement FAILED: {e}", exc_info=True)
        return None

In [ ]:
zResult = showSettlement(dummySession,dummySettlment)

In [ ]:
zResult

In [ ]:
aBoard = globalStore.active_board(dummySession['userid']).board
aBoard

In [ ]:
@patch
def project_region(region: HexRegion, target_grid: HexGrid, c2f: dict = None) -> HexRegion:
    """Project a coarse region onto a target grid via c2f mapping.
    If c2f is None (identity), returns the region unchanged."""
    if c2f is None:
        return region
    fine_hexes = set()
    for coarse_idx in region.hexes:
        if coarse_idx in c2f:
            fine_hexes.update(c2f[coarse_idx])
    return HexRegion(hexes=fine_hexes, hexGrid=target_grid)


def _map_point(coarse_idx: int, c2f: dict = None) -> int:
    """Map a single coarse index to a fine index (picks middle representative).
    If c2f is None (identity), returns coarse_idx unchanged."""
    if c2f is None:
        return coarse_idx
    fs = c2f.get(coarse_idx)
    if not fs:
        return -1
    return fs[len(fs) // 2]

In [ ]:
@patch
def settlementTouchOverlay(self: GameBoard, terrain: Terrain = None,
                      c2f: dict = None, scale: float = 2.0,attrs= None) -> str:
    """Settlement markers using king pieces with flag patterns.

       Place a chess king at center. attrs: dict of HTML/HTMX attributes for interactivity.
    For instance:
    Then usage looks like:
    
flag.kingPiece(center, scale=1.5, attrs={
    'hx-get': '/piece/king/clicked',
    'hx-target': '#game-panel',
    'hx-swap': 'innerHTML',
    'data-piece': 'king',
    'data-player': flag.name
})

    
    """
    terrain = terrain or self.terrain
    grid = terrain.hexGrid
    num_hexes = len(grid.hexes)

    overlay = ""
    for i, country in enumerate(self.kingdoms):
        if country.flag is None:
            continue

        # Register flag pattern
        pat_name = f"settle_pat_{country.countryId}"
        pat = country.flag.flagPattern(pat_name, scale=0.1)
        grid.builder.add_definition(pat)
        pat_fill = f"url(#{pat_name})"

        for j, s in enumerate(country.settlements):
            if s.location is None:
                continue
            local_idx = _map_point(s.location, c2f)
            if local_idx < 0 or local_idx >= num_hexes:
                continue
            coords = grid.hexes[local_idx].center
            piece_id = f"king_{country.countryId}_{j}"
            overlay += country.flag.kingPiece(
                MapCord(coords.x, coords.y),
                scale=scale, piece_id=piece_id, fill=pat_fill, attrs=attrs
            )

    return overlay


In [ ]:
zTerr = zResult.terrain
zTerr.hexGrid.builder.layers = []
zTerr.colorMap()
zTerr.hexGrid.update()
c2f = zResult.invert_mapper()

borders = aBoard.countries_overlay(zResult.terrain, c2f)
names   = aBoard.names_overlay(zResult.terrain, c2f)
settle  = aBoard.settlementOverlay(zResult.terrain, c2f)
#print(names)
zTerr.hexGrid.builder.adjust("borders",borders)
zTerr.hexGrid.builder.adjust("names",names)
zTerr.hexGrid.builder.adjust("settle",settle)
#zTerr.hexGrid.builder.show()

I think we have enough pieces to get started on building the settlement map route. maybe clicking on the settlement would bring up a detail page we could put in either the right or left panel.

I think right panel and just an HTMX partial. This is a great example of when we are staying within a mode.

In [ ]:
??ZoomResult

In [ ]:
!tail -10 base.text

Any ideas how to fix showSettlement? can we add better logging?

So I think Settlement name + health/size stats would make sense. In theory we would have the list citzens on the left panel.

## Settlment Map

@rt("/settlement/{id}")
def settlementMap(session, id: str, rings: int = None):
    """Settlement map view — zoomed terrain with overlays."""
    uid = ensure_user(session)
    
    # Read from session if not provided, default 5
    if rings is None:
        rings = session.get('settlement_rings', 5)
    else:
        rings = max(3, min(12, rings))  # clamp
    session['settlement_rings'] = rings

    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        # Zoom into settlement region
        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()

        c2f = result.invert_mapper()
        zoomed.terrainCream()

        builder = zoomed.hexGrid.builder
        builder.adjust("borders", board.countries_overlay(zoomed, c2f))
        builder.adjust("names", board.names_overlay(zoomed, c2f))

        if result.basins:
            builder.adjust("water", result.basins.draw_watersheds())

        # Settlement king piece — clickable, targets right panel
        settle_attrs = {
            'hx-get': f'/settlement_detail/{id}',
            'hx-target': '#debug-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        settle_overlay = board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs)
        builder.adjust("settlements", settle_overlay)

        # Kingdom info for breadcrumbs
        k = next((k for k in board.kingdoms if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"

        # --- Map div (primary swap target) ---
        map_div = Div(
            Div(
                A("🌍 World", hx_get="/showMap", hx_target="#map",
                  cls="btn btn-sm btn-outline"),
                Span(" → "),
                A(kingdom_name, hx_get=f"/kingdom/{place.owner_id}",
                  hx_target="#map", cls="btn btn-sm btn-outline"),
                Span(" → "),
                Span(place.name or "Settlement", cls="font-bold"),
                cls="flex items-center gap-2 p-2"
            ),
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )

        # --- Left panel (OOB swap) ---
        citizen_items = []
        for c in place.citizens:
            citizen_items.append(
                Li(A(f"🧑 {c.id[:8]}… HP:{c.health}/{c.max_health}",
                     hx_get=f"/piece_detail/{c.id}",
                     hx_target="#debug-panel",
                     cls="link link-hover"),
                   cls="text-sm")
            )
        if not citizen_items:
            citizen_items = [Li("No citizens yet", cls="text-sm opacity-50")]

        left_panel = Div(
            H4(place.name or "Settlement", cls="font-bold text-lg"),
            P(f"Kingdom: {kingdom_name}", cls=TextPresets.muted_sm),
            Divider(),
            P("Citizens", cls=TextPresets.bold_sm),
            Ul(*citizen_items, cls="space-y-1"),
            Divider(),
            A("← Back to Kingdom", hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="btn btn-outline btn-sm"),
            id="left-panel",
            hx_swap_oob="true",
            cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
        )

        return map_div, left_panel

    except Exception as e:
        logging.error(f"settlement route FAILED: {e}", exc_info=True)
        return P(f"Error loading settlement: {e}")


# ── Updated settlement_map — respects overlay toggles ────────

@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    """Map partial — renders zoomed terrain with active overlays only."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active: return P("No active game")

    rings = session.get('settlement_rings', 5)
    overlays = get_overlays(session)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        builder = zoomed.hexGrid.builder

        # ── Conditional overlays ──
        if 'cream' in overlays:
            zoomed.terrainCream()

        builder.adjust("borders", board.countries_overlay(zoomed, c2f))  # always show borders

        if 'names' in overlays:
            builder.adjust("names", board.names_overlay(zoomed, c2f))

        if 'rivers' in overlays and result.basins:
            builder.adjust("water", result.basins.draw_watersheds())

        if 'settlements' in overlays:
            settle_attrs = {
                'hx-get': f'/settlement_detail/{id}',
                'hx-target': '#debug-panel',
                'hx-swap': 'innerHTML',
                'style': 'cursor: pointer;',
            }
            builder.adjust("settlements",
                           board.settlementTouchOverlay(zoomed, c2f, attrs=settle_attrs))

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


In [ ]:
@rt("/settlement_map/{id}")
def settlement_map(session, id: str, rings: int = None):
    """Map partial — renders zoomed terrain with active overlays."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")

    if rings is not None:
        rings = max(3, min(12, rings))
        session['settlement_rings'] = rings
    else:
        rings = session.get('settlement_rings', 5)

    overlays = get_overlays(session)

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        terrain = board.terrain

        region = place.region(terrain.hexGrid, rings=rings)
        if len(region.hexes) == 0:
            return P("Settlement region is empty", id="map")

        result = active.cover.zoom_region_fast(region, compute_weather=True)
        zoomed = result.terrain
        zoomed.hexGrid.adjustRadius(terrain.hexGrid.radius)
        zoomed.hexGrid.builder.layers = []
        zoomed.colorMap()
        zoomed.hexGrid.update()
        c2f = result.invert_mapper()

        settle_attrs = {
            'hx-get': f'/settlement_detail/{id}',
            'hx-target': '#right-panel',
            'hx-swap': 'innerHTML',
            'style': 'cursor: pointer;',
        }
        zoomed.apply_overlays(result, board, c2f, overlays, settle_attrs)

        return Div(
            HexTouchMap(zoomed.hexGrid, cls="w-full h-full"),
            id="map"
        )
    except Exception as e:
        logging.error(f"settlement_map FAILED: {e}", exc_info=True)
        return P(f"Error: {e}", id="map")


### bottom panel

In [ ]:
# ── Controls route — HexLegend toggle bar ────────────────────

@rt("/settlement_controls/{id}")
def settlement_controls(session, id: str):
    """Bottom bar with toggleable overlay hex swatches."""
    active = get_overlays(session)

    hexes, values = [], []
    for key, cfg in OVERLAY_CONFIG.items():
        is_on = key in active
        fill   = cfg['fill']   if is_on else '#888'
        stroke = cfg['stroke'] if is_on else '#555'
        opacity = '1.0' if is_on else '0.35'

        style = StyleCSS(f"ov_{key}", fill=fill, stroke=stroke,
                         stroke_width=2, opacity=opacity)
        label = cfg['label'] + (' ✓' if is_on else '')
        h = Hex(radius=15, center=MapCord(0, 0), style=style, label=label)
        hexes.append(h)
        values.append(key)

    return HexLegend(
        hexes, name="overlay", values=values,
        hx_post=f"/toggle_overlay/{id}",
        hx_target="#map", hx_swap="outerHTML",
        direction="row", size=30, font_size=11,
        gap=4, item_gap=16,
    )


## On the side

@rt("/settlement/{id}")
def settlement(session, id: str):
    """Full-page shell for settlement view — panels load via HTMX."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Settlement",
        Div(
            # Left sidebar — breadcrumbs + citizens
            Div(
                id="left-panel",
                hx_get=f"/settlement_left/{id}",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),

            # Main area
            Div(
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map",
                    hx_get=f"/settlement_map/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                Div(id="map-controls", cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),

            # Right sidebar — detail card
            Div(
                H4("Settlement", cls="text-sm font-bold mb-2"),
                Div(id="debug-panel",
                    hx_get=f"/settlement_detail/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


Do I need to wire settlement_controls into something? Should I redo def settlement(session, id: str):

In [ ]:
@rt("/settlement_left/{id}")
def settlement_left(session, id: str):
    """Left panel partial — breadcrumbs + citizen list."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    try:
        place = globalStore.settlement_from_id(id)
        board = active.board
        k = next((k for k in board.kingdoms if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"
        name = place.name or "Settlement"

        breadcrumbs = Div(
            A("🌍 World", hx_get="/showMap", hx_target="#map",
              cls="link link-hover text-sm"),
            Span(" → ", cls="opacity-50"),
            A(kingdom_name, hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="link link-hover text-sm"),
            Span(" → ", cls="opacity-50"),
            Span(name, cls="text-sm font-bold"),
            cls="flex flex-wrap items-center gap-1 mb-4"
        )

        citizen_items = []
        for c in getattr(place, 'citizens', []):
            citizen_items.append(
                Li(A(f"🧑 {c.id[:8]}… HP:{c.health}/{c.max_health}",
                     hx_get=f"/piece_detail/{c.id}",
                     hx_target="#right-panel",
                     cls="link link-hover"),
                   cls="text-sm")
            )
        if not citizen_items:
            citizen_items = [Li("No citizens yet", cls="text-sm opacity-50")]

        return Div(
            breadcrumbs,
            Divider(),
            H4(name, cls="font-bold text-lg"),
            P(f"Kingdom: {kingdom_name}", cls=TextPresets.muted_sm),
            Divider(),
            P("Citizens", cls=TextPresets.bold_sm),
            Ul(*citizen_items, cls="space-y-1"),
            Divider(),
            A("← Back to Kingdom",
              hx_get=f"/kingdom/{place.owner_id}",
              hx_target="#map", cls="btn btn-outline btn-sm"),
        )
    except Exception as e:
        logging.error(f"settlement_left FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


In [ ]:
def flag_hex(flag, radius=25):
    """Hex with flag pattern fill, built via SVGBuilder."""
    pad = 4
    size = radius * 2 + pad * 2
    center = MapCord(size / 2, size / 2)

    builder = SVGBuilder()
    builder.width = size
    builder.height = size

    # Register flag pattern as a fill
    pat_name = f"fh_{flag.name}"
    builder.add_definition(flag.flagPattern(pat_name, scale=0.08))
    pat_style = StyleCSS(pat_name, fill=f"url(#{pat_name})",
                         stroke=flag.comp, stroke_width=3)
    builder.add_style(pat_style)

    h = Hex(radius=radius, center=center, style=pat_style)
    builder.adjust("hex", h.svg())

    return Div(NotStr(builder.xml()))


In [ ]:
@rt("/settlement_detail/{id}")
def settlement_detail(session, id: str):
    """Right-panel summary card for a settlement."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game")

    try:
        place = globalStore.settlement_from_id(id)
        k = next((k for k in active.board.kingdoms
                  if k.countryId == place.owner_id), None)
        kingdom_name = k.countryName if k else f"Kingdom {place.owner_id}"
        name = place.name or "Settlement"

        dark = k.flag.darkPrimary if k and k.flag else "#333"
        current_rings = session.get('settlement_rings', 5)

        header_parts = []
        if k and k.flag:
            header_parts.append(flag_hex(k.flag))
        header_parts.append(
            Div(
                H3(name, cls="font-bold",
                   style=f"font-family: 'Cinzel', serif; color: {dark};"),
                P(kingdom_name, cls=TextPresets.muted_sm,
                  style=f"font-family: 'Cinzel', serif; color: {dark}; opacity: 0.7;"),
            )
        )

        radius_control = Div(
            P("View Radius", cls=TextPresets.bold_sm),
            Input(type="range", name="rings", id="rings-slider",
                  min="3", max="12", value=str(current_rings),
                  hx_get=f"/settlement_map/{id}",
                  hx_target="#map",
                  hx_swap="outerHTML",
                  hx_trigger="change",
                  hx_include="#rings-slider",
                  cls="uk-range w-full"),
            P(f"{current_rings} rings", id="rings-label", cls=TextPresets.muted_sm),
            Script("""
                me('#rings-slider').on('input', ev => {
                    me('#rings-label').textContent = ev.target.value + ' rings';
                });
            """),
            cls="space-y-2"
        )

        return Div(
            Link(rel="stylesheet",
                 href="https://fonts.googleapis.com/css2?family=Cinzel:wght@400;700&display=swap"),
            DivLAligned(*header_parts, cls="gap-3"),
            Divider(),
            Div(
                DivFullySpaced(Span("Health", cls=TextPresets.muted_sm),
                               Span(f"{place.health}/100")),
                Progress(value=str(place.health), max="100"),
                DivFullySpaced(Span("Size", cls=TextPresets.muted_sm),
                               Span(str(place.size))),
                DivFullySpaced(Span("Citizens", cls=TextPresets.muted_sm),
                               Span(str(len(place.citizens)))),
                DivFullySpaced(Span("Location", cls=TextPresets.muted_sm),
                               Span(f"Hex {place.location}")),
                cls="space-y-2"
            ),
            Divider(),
            radius_control,
            cls="space-y-3"
        )
    except Exception as e:
        logging.error(f"settlement_detail FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


## Put it together

In [ ]:
@rt("/settlement/{id}")
def settlement(session, id: str):
    """Full-page shell for settlement view — panels load via HTMX."""
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return RedirectResponse('/', status_code=303)

    return Titled("Settlement",
        Div(
            # Left sidebar — breadcrumbs + citizens
            Div(
                id="left-panel",
                hx_get=f"/settlement_left/{id}",
                hx_trigger="load",
                hx_indicator="#spinner",
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),

            # Main area
            Div(
                Loading(cls=(LoadingT.spinner, LoadingT.lg),
                        htmx_indicator=True, id="spinner"),
                Div(id="map",
                    hx_get=f"/settlement_map/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner",
                    cls="flex-1 overflow-auto"),
                # ← wired up here
                Div(id="map-controls",
                    hx_get=f"/settlement_controls/{id}",
                    hx_trigger="load",
                    cls="h-16 bg-base-300 p-2 flex-shrink-0"),
                cls="flex-1 flex flex-col items-center justify-center relative"
            ),

            # Right sidebar — detail card
            Div(
                H4("Settlement", cls="text-sm font-bold mb-2"),
                Div(id="right-panel",
                    hx_get=f"/settlement_detail/{id}",
                    hx_trigger="load",
                    hx_indicator="#spinner"),
                cls="w-64 bg-base-200 p-4 flex-shrink-0 overflow-y-auto"
            ),
            cls="flex h-screen"
        )
    )


In [ ]:
webMe(settlement_detail(dummySession, dummySettlment))

Did I build out everything correctly?

Did I build out everything correctly?

can you build what I am mising in settlement_detail, settlement_map, settlement_left and toggle_overlay 

Maybe we could have a slider that would set settlementdetail radius. We probably want to put this into a cookie for the session.

In [ ]:
webMe(settlement(dummySession, dummySettlment,rings = 2))

In [ ]:
??HexLegend

so I want to build out the map-controls part of our main setttlement page using a HexLegend at the bottom. We have several overlays that we would like to toggle settlement/names/cream/rivers that we should store in the cookie but then use when we are rendering the map.

I am going to work through some thoughts

## Modified Gosper
so the rules on Piece is going to be reworked as a list of ints with the following
0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
4 -> settle or siege # reserved for figuring out how to conquer settlements
5 -> forward # move on place forward (depending on movement cost). If you can't move forward (ie you hit water) rotate left. if it is there is another piece in the way pause   
6 -> rotate left
7 -> rotate right


## Movement costs

we have a movement cost formula
```python
@patch
def movement_cost(self: GameBoard, from_pos: HexPosition, to_pos: HexPosition, origin: int) -> float:
    """Calculate cost to move from one hex to another using HexPositions.
    
    Args:
        from_pos: Starting HexPosition
        to_pos: Destination HexPosition
        origin: Origin index for converting HexPosition to grid index
    
    Returns infinity for invalid moves (water, out of bounds).
    """
    terrain = self.terrain
    grid = terrain.hexGrid
    
    # Convert to indices to check terrain
    to_idx = grid.hexposition_to_index(to_pos, origin)
    from_idx = grid.hexposition_to_index(from_pos, origin)
    
    # Check bounds
    if to_idx < 0 or to_idx >= len(terrain.elevations):
        return float('inf')
    
    # Can't cross water
    if terrain.elevations[to_idx] < 1:
        return float('inf')
    
    from_elev = terrain.elevations[from_idx]
    to_elev = terrain.elevations[to_idx]
    
    # Base cost
    base_cost = 1.0
    
    # Elevation gain penalty (climbing is expensive)
    elev_diff = to_elev - from_elev
    if elev_diff > 0:
        # Exponential penalty for climbing
        base_cost += elev_diff * 2.0
    
    return base_cost
```
Which we will need to figure out how to use for a turn. movement_range will tell how many moves we can do. Lets make turns free (including turns from hitting the ocean). the first 4 counts as a single movement cost.

## food and diet
we are going to add three fields 
1. which is the amount of food a piece has
2. its diet. how much does it eat per turn If it rounds out of food it will start to lose health. 
3. food capacity which is how much total food it can carry.

## deprecate
we don't need memory since that will be in rules.

## settle
We are going to keep these around even though I haven't figure them out
harvest_goal: Resources = None
settle_progress: int = 0
settle_threshold: int = 3
sight = 3

## The things we carry
Lets start thinking about the rules editor for a piece
1. we are going to want a list of rules that are assigned and be able to add and delete rows
2. we are going to want an edit point where we can insert new items
3. we are going to want a HexDragMap that we can add items. The path we make needs to be converted into instructions on how we get there.
4. we need an apply button and the number of rounds. This will be our main map showing where the piece is going to move each turn for the number of turns. I envision this as a series of arrows and circles.
a. arrows are for movement. straight line if single cost squiggle line if multiple cost. the color of the arrow is how aggressive it is.
b. circles are for the pause actions (0 -> pause
1 -> defend # in defend mode we get more aggressive
2 -> harvest # gather food from the hex
3 -> give   # give food to either a neighbor piece or a settlement
) I think red should be for defend.

eventually we are going to want a drop down of a list of common rule sets (patrol around a circle. go out and harvest and comeback, attack by first flanking left). 
we will certainly want to have a hex destination that returns a set of rules on how to get there (hello djisktra)

I need to be clearer on some definitons
1. `round` is for all player and all pieces. Each player gives the set of orders it wants its pieces to do. We need to build out how to carry them out, but I am tempted just to start with the kindergaren model of everybody does what it wants
2. `turn` is equivalent to a round, but just on a single piece. `movement_cost` is how many things it can do in a turn. Some (like rotation) are free. Others like mmoving up a hill might cost a couple of movements.

It can be the case that rule ends and there are spare items. So if the last item in the list is one of the non movement items. ie (like pause) then we just pause the remaining amount of time. if it is a movement (or rotate) then we circle back to the begining.

## Pulling Ranks
so things are prioritzed in chess order ie kings move first and eat first. then queens. This way if a king will move to a spot before a pawn would

I am going to have the flow of the hex be the amount you can harvest. Settlments are going to have build queues. I need to figure out the mechanics of adding settlements whether there are pre existing locations or it behavior on a item

**7. Food economy**

Ill need a `hex_food_yield(terrain, idx) -> float` keyed off climate and flow.



**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times but each time if the chance to move becomes available



What would you add to this or worry about?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great design doc. Here's what I'd add or flag, roughly in priority order:

**1. Turn execution model — the biggest open question**

You've defined *what* each instruction does, but not *how many* execute per turn. Two options:
- **One instruction per turn** — simple but slow; a piece going 4 hexes takes 4+ turns
- **Execute until movement_range exhausted** — more interesting, but then non-movement actions (0–3) need a cost or a "one per turn" cap, otherwise a piece could harvest forever in one turn

I'd suggest: each turn, execute instructions until you've spent `movement_range` worth of movement *or* performed one non-movement action (whichever comes first). Forward costs movement points, rotates are free, and 0/1/2/3 each end the turn immediately.

**2. End-of-list behavior**

When `cursor` hits the end of the rules list — loop back to 0? Stop and idle? This determines whether patrols are naturally cyclic. I'd default to **loop** with a boolean `patrol: bool = True` flag on the piece.

**3. Forward-blocked spin trap**

Instruction 4 says "rotate left if water, pause if piece." But imagine a piece in a cove with water on 5 sides — it rotates 5 times and hits water again. You need a `max_rotates = 5` guard that converts to a pause if all directions are blocked.

**4. Give targeting**

Instruction 3 needs a target. Options:
- Always give to nearest friendly piece/settlement within `sight` range
- Encode the target as a parameter (makes rules more complex)

I'd start with "give to nearest settlement if adjacent, else nearest friendly piece if adjacent, else pause."

**5. Path → rules conversion is the hard part**

The drag-to-rules conversion needs to track facing at each step:

```python
def path_to_rules(positions: list[HexPosition], start_facing: int) -> list[int]:
    """Convert a hex path into forward/rotate instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    for i in range(len(positions) - 1):
        delta = positions[i+1] - positions[i]
        target_dir = next(j for j, d in enumerate(dirs) if d == delta)
        # Shortest rotation
        diff = (target_dir - facing) % 6
        if diff <= 3:
            rules.extend([6] * diff)       # rotate right
        else:
            rules.extend([5] * (6 - diff)) # rotate left
        rules.append(4)  # forward
        facing = target_dir
    return rules
```

This is a good first pass — Dijkstra for "go to hex X" would wrap this with pathfinding.

**6. Dry-run simulation**

Your visualization (arrows + circles for N rounds) requires simulating the rules forward *without* mutating game state. I'd build a `simulate_piece(piece, board, n_turns) -> list[SimStep]` that returns positions, facings, and action types per turn. The overlay renderer then draws from that list.

**7. Food economy**

You'll need a `hex_food_yield(terrain, idx) -> float` keyed off terrain type/elevation. Probably: plains high, forests medium, mountains low, water zero. Without this, harvest does nothing.

**8. Fields to add to Piece**

Based on your notes:

```python
# Add these
food: float = 0.0
diet: float = 1.0          # food consumed per turn
food_capacity: float = 10.0
rules: list[int] = field(default_factory=list)
patrol: bool = True         # loop rules?

# Deprecate
# memory: float = 0.8      # replaced by explicit rules
# rule: str = ""            # replaced by rules: list[int]
```

Keep `cursor` — it still tracks position in the new `rules` list.

**9. Collision resolution order**

Two pieces moving into the same hex on the same turn — who wins? You'll need this eventually. Simplest: resolve in piece-creation order, first mover claims the hex, second gets a pause. Or: simultaneous collision = both pause.

**Things I wouldn't worry about yet:** common rule presets (that's UI sugar), flanking mechanics, and the settle workflow. Get the core turn loop + rules editor working first, then layer those on.

I addressed some of these

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good progress. Here's where I see things standing:

**Solid and clear now:**
- The 0–7 instruction set is clean
- Food/diet/capacity as three fields
- Movement cost model (rotations free, first 4 instructions cost 1 each, forward costs terrain-dependent)
- End-of-list behavior (pause-tail → idle, move-tail → loop)
- Priority by piece type (chess rank order)

**Still needs nailing down:**

**Turn budget accounting.** You say "the first 4 counts as a single movement cost" — I read that as instructions 0–3 each consume 1 movement point. But can a piece do *multiple* non-movement actions in one turn? E.g. if `movement_range = 4`, can it harvest 4 times? I'd suggest: non-movement actions cost 1 point *and* end the turn (so max one per turn). Otherwise harvest loops become overpowered.

**Instruction 5 (forward) ambiguity.** The "rotate left if water" fallback effectively combines two instructions (forward + conditional rotate). That makes the spin trap real — you acknowledged it but didn't settle on a fix. I'd go with: forward attempts once, if blocked by water rotate left once and *stop* (spend 0 movement, but cursor stays on the forward instruction). Next turn it tries again from the new facing. This avoids the 5-rotation spin and keeps each instruction atomic.

**Instruction 3 (give) targeting.** Who receives? Nearest adjacent friendly, or does the facing matter? Simplest: give to whatever friendly piece or settlement is in the hex you're facing. If empty, pause instead.

**Instruction 4 (settle/siege).** You flagged this as "reserved" — just make sure the cursor advances past it for now so it doesn't block rule execution. Treat it as a pause until you define the mechanic.

**Rules encoding.** Since `rule: str` is being replaced by `rules: list[int]`, you'll want encode/decode for persistence. A comma-separated string of single digits is compact: `"5,5,6,5,5,7,2"`. Cursor stays as an int index into that list.

Want to start building the updated `Piece` dataclass with these new fields, or jump to the rules editor UI?

I think I got this in

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — and looking at the `!cat` output earlier, the new `data.py` already has it all: `Instruction` enum, `rules: List[int]`, `patrol: bool`, `food/diet/food_capacity`, and the updated encode/decode. Looks clean.

So the natural next step is the **rules editor UI** for `piece_detail`. You've got the facing selector ready. The rules editor needs:

1. **Rule list** — a reorderable list of the current `piece.rules`, each row showing the instruction name (from `Instruction` enum) with a delete button
2. **Append controls** — buttons or a small hex palette for each of the 8 instructions
3. **HexDragMap path → rules** — drag a path, converts to `FORWARD`/`ROT_L`/`ROT_R` via the `path_to_rules` logic we sketched
4. **Simulate preview** — run the rules forward N turns and draw arrows/circles on the map

Which piece do you want to build first? The list editor is the simplest starting point, or the drag-to-path converter is the most interesting.

the drag-to-path converter. We could pratice with a dummy piece maybe a rook.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's build a standalone demo. The core challenge is that `HexDragMap` gives us a comma-separated list of hex indices — we need to convert those back to `HexPosition` deltas and then into `FORWARD`/`ROT_L`/`ROT_R` instructions.

```python
from HexMagic.game.data import Piece, PieceType, Instruction

# ── 1. Path → rules converter ─────────────────────────────────

def path_to_rules(hex_indices: list[int], grid: HexGrid, start_facing: int = 0) -> list[int]:
    """Convert a list of hex grid indices (from drag) into Instruction ints."""
    if len(hex_indices) < 2:
        return []

    dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] = indices 0–5

    # Deduplicate consecutive duplicates (drag can repeat hexes)
    clean = [hex_indices[0]]
    for idx in hex_indices[1:]:
        if idx != clean[-1]:
            clean.append(idx)

    rules = []
    facing = start_facing

    for i in range(len(clean) - 1):
        from_pos = grid.index_to_hexposition(clean[i],   clean[0])
        to_pos   = grid.index_to_hexposition(clean[i+1], clean[0])
        delta    = to_pos - from_pos

        # Find which direction matches the delta
        try:
            target_dir = next(j for j, d in enumerate(dirs) if d == delta)
        except StopIteration:
            # Non-adjacent hex in drag — skip
            continue

        # Shortest rotation from facing to target_dir
        diff = (target_dir - facing) % 6
        if diff == 0:
            pass                                        # already facing right
        elif diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        rules.append(Instruction.FORWARD.value)
        facing = target_dir

    return rules


# ── 2. Rules → readable labels ────────────────────────────────

_INSTR_LABELS = {
    0: ("⏸ Pause",   "#94a3b8"),
    1: ("🛡 Defend",  "#ef4444"),
    2: ("🌾 Harvest", "#22c55e"),
    3: ("🎁 Give",    "#a855f7"),
    4: ("🏰 Settle",  "#f59e0b"),
    5: ("→ Forward", "#3b82f6"),
    6: ("↺ Rot L",   "#64748b"),
    7: ("↻ Rot R",   "#64748b"),
}

def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence."""
    if not rules:
        return P("No rules yet — drag a path above", cls="text-sm opacity-50")

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    return Div(
        P(f"{len(rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=f"rules-{piece_id[:8]}",
    )


# ── 3. The drag map widget ─────────────────────────────────────

def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """
    HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display.
    """
    base    = StyleCSS("pe-hex",    fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",    fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label",  fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Mark piece position
    grid.hexes[mid].style = current
    grid.hexes[mid].label = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    # Draw facing arrow from center
    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        arrow_layer = grid.arrow(mid, facing_idx,
                                 style=StyleCSS("pe-arrow", stroke="#f59e0b",
                                                stroke_width=2))
        grid.builder.adjust("facing", arrow_layer)

    grid.update()

    widget_id = f"drag-editor-{piece.id[:8]}"

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=widget_id,
            style="touch-action:none;"
        ),
        rules_display(piece.rules, piece.id),
        cls="space-y-2"
    )


# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)


# ── 5. Quick demo with a dummy Rook ───────────────────────────

dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    rules      = [],
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))
```

The flow is:

1. You see a dark hex grid with an amber **R** hex in the center and an arrow showing the rook's current facing
2. Drag across hexes — `HexDragMap` collects the indices
3. POST fires to `/piece_path/{id}/{facing}` — `path_to_rules` computes the minimal rotate + forward sequence for your exact drag path
4. Instruction badges appear below in colour-coded chips

The one thing to check before wiring to a real piece: `piece_drag_editor` and `/piece_path/` need to agree on `rings` and `radius` since the grid is reconstructed server-side to do the index→position math. Want to extract those as constants or pass them as hidden form fields?

In [ ]:
from HexMagic.game.data import Piece, PieceType, Instruction

# ── 1. Path → rules converter ─────────────────────────────────

def path_to_rules(hex_indices: list[int], grid: HexGrid, start_facing: int = 0) -> list[int]:
    """Convert a list of hex grid indices (from drag) into Instruction ints."""
    if len(hex_indices) < 2:
        return []

    dirs = HexPosition.directions()   # [SW, W, NW, NE, E, SE] = indices 0–5

    # Deduplicate consecutive duplicates (drag can repeat hexes)
    clean = [hex_indices[0]]
    for idx in hex_indices[1:]:
        if idx != clean[-1]:
            clean.append(idx)

    rules = []
    facing = start_facing

    for i in range(len(clean) - 1):
        from_pos = grid.index_to_hexposition(clean[i],   clean[0])
        to_pos   = grid.index_to_hexposition(clean[i+1], clean[0])
        delta    = to_pos - from_pos

        # Find which direction matches the delta
        try:
            target_dir = next(j for j, d in enumerate(dirs) if d == delta)
        except StopIteration:
            # Non-adjacent hex in drag — skip
            continue

        # Shortest rotation from facing to target_dir
        diff = (target_dir - facing) % 6
        if diff == 0:
            pass                                        # already facing right
        elif diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        rules.append(Instruction.FORWARD.value)
        facing = target_dir

    return rules


# ── 2. Rules → readable labels ────────────────────────────────

_INSTR_LABELS = {
    0: ("⏸ Pause",   "#94a3b8"),
    1: ("🛡 Defend",  "#ef4444"),
    2: ("🌾 Harvest", "#22c55e"),
    3: ("🎁 Give",    "#a855f7"),
    4: ("🏰 Settle",  "#f59e0b"),
    5: ("→ Forward", "#3b82f6"),
    6: ("↺ Rot L",   "#64748b"),
    7: ("↻ Rot R",   "#64748b"),
}

def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence."""
    if not rules:
        return P("No rules yet — drag a path above", cls="text-sm opacity-50")

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    return Div(
        P(f"{len(rules)} instructions", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        id=f"rules-{piece_id[:8]}",
    )


# ── 3. The drag map widget ─────────────────────────────────────

def piece_drag_editor(piece: Piece, rings: int = 3, radius: int = 35):
    """
    HexDragMap centered on piece location.
    Drag a path → POST hex_ids → convert to rules → display.
    """
    base    = StyleCSS("pe-hex",    fill="#1e293b", stroke="#475569", stroke_width=1)
    current = StyleCSS("pe-cur",    fill="#f59e0b", stroke="#fbbf24", stroke_width=2)
    lbl     = StyleCSS("pe-label",  fill="#e2e8f0", font_size="11px")

    grid = HexGrid.centered(rings, radius=radius, style=base)
    for s in [current, lbl]:
        grid.builder.add_style(s)

    mid = grid.middle

    # Mark piece position
    grid.hexes[mid].style = current
    grid.hexes[mid].label = piece.piece_type.value[0].upper()
    grid.hexes[mid].labelStyle = lbl.name

    # Draw facing arrow from center
    facing_dir = HexPosition.directions()[piece.facing % 6]
    facing_idx = grid.hexposition_to_index(facing_dir, mid)
    if facing_idx >= 0:
        arrow_layer = grid.arrow(mid, facing_idx,
                                 style=StyleCSS("pe-arrow", stroke="#f59e0b",
                                                stroke_width=2))
        grid.builder.adjust("facing", arrow_layer)

    grid.update()

    widget_id = f"drag-editor-{piece.id[:8]}"

    return Div(
        H5("Drag a patrol path", cls="text-sm font-bold mb-1"),
        P("Drag across hexes to generate movement instructions",
          cls="text-xs opacity-50 mb-2"),
        
        ##
        HexDragMap(
            grid,
            on_drag={
                "hx_post":   f"/piece_path/{piece.id}/{piece.facing}",
                "hx_target": f"#rules-{piece.id[:8]}",
                "hx_swap":   "outerHTML",
            },
            id=widget_id,
            # removed style="touch-action:none;" — HexDragMap adds it already
        ),

        ##
        rules_display(piece.rules, piece.id),
        cls="space-y-2"
    )


# ── 4. Route that receives the drag ───────────────────────────

@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, persist, return badge display."""
    form = await request.form()
    raw  = form.get("hex_ids", "")

    if not raw:
        return Div(P("No path received", cls="text-sm opacity-50"),
                   id=f"rules-{piece_id[:8]}")

    hex_ids = [int(x) for x in raw.split(",") if x.strip()]

    # We need the grid to convert indices → HexPositions.
    # Use a fresh centered grid matching piece_drag_editor's rings/radius.
    rings  = 3
    radius = 35
    dummy_style = StyleCSS("x", fill="white")
    grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

    new_rules = path_to_rules(hex_ids, grid, start_facing=facing)

    # Persist to DB if we can find the piece
    try:
        rec = globalStore.pieces[piece_id]
        if rec:
            existing = [int(x) for x in (rec.get("rules") or "").split(",") if x]
            merged = existing + new_rules   # append; swap for replace if preferred
            globalStore.pieces.upsert(
                {"id": piece_id, "rules": ",".join(str(r) for r in merged)},
                pk="id"
            )
            new_rules = merged
    except Exception:
        pass   # demo mode — just show without saving

    dummy = Piece(id=piece_id, rules=new_rules)
    return rules_display(dummy.rules, piece_id)


# ── 5. Quick demo with a dummy Rook ───────────────────────────

dummy_rook = Piece(
    id   = "demo-rook-0000-0000-000000000000",
    piece_type = PieceType.ROOK,
    facing     = 4,   # starts facing E
    rules      = [],
)

webMe(Div(
    H3("Rook Rule Editor"),
    piece_drag_editor(dummy_rook),
    cls="p-4 space-y-4 bg-slate-900 text-white"
))


Help?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It rendered! The hex grid, the amber R, and the facing arrow all look correct. The drag JS is wired up too.

The catch is that `webMe()` renders a static iframe — the HTMX POST from the drag won't reach your server. So you can't test the full drag → rules loop in preview mode.

Two ways forward:

**1. Test `path_to_rules` directly** — verify the conversion logic works:

```python
# Simulate a drag path: center → E → NE → NE
grid = HexGrid.centered(3, radius=35, style=StyleCSS("x", fill="white"))
mid = grid.middle  # 24

# Manually pick a path: center, then one E, then one NE
e_idx  = grid.hexposition_to_index(HexPosition.E, mid)
ne_idx = grid.hexposition_to_index(HexPosition.NE, mid)
ne2    = grid.hexposition_to_index(HexPosition.NE + HexPosition.NE, mid)

drag_path = [mid, e_idx, ne_idx, ne2]
print(f"Drag indices: {drag_path}")

rules = path_to_rules(drag_path, grid, start_facing=4)  # facing E
labels = [_INSTR_LABELS[r][0] for r in rules]
print(f"Rules: {rules}")
print(f"Labels: {labels}")
```

**2. Wire it into the live server** — add the route to a page you can actually navigate to:

```python
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    return Titled("Piece Editor",
        Div(
            H3("Rook Rule Editor"),
            piece_drag_editor(dummy_rook),
            cls="p-4 space-y-4"
        )
    )
```

Then visit `/piece_editor_demo` in the browser and drag away — the HTMX round-trip will work there.

I'd start with option 1 to make sure the conversion math is right, then option 2 to see the full interactive loop.

In [ ]:
# Simulate a drag path: center → E → NE → NE
grid = HexGrid.centered(3, radius=35, style=StyleCSS("x", fill="white"))
mid = grid.middle  # 24

# Manually pick a path: center, then one E, then one NE
e_idx  = grid.hexposition_to_index(HexPosition.E, mid)
ne_idx = grid.hexposition_to_index(HexPosition.NE, mid)
ne2    = grid.hexposition_to_index(HexPosition.NE + HexPosition.NE, mid)

drag_path = [mid, e_idx, ne_idx, ne2]
print(f"Drag indices: {drag_path}")

rules = path_to_rules(drag_path, grid, start_facing=4)  # facing E
labels = [_INSTR_LABELS[r][0] for r in rules]
print(f"Rules: {rules}")
print(f"Labels: {labels}")


Is it working?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! Let me trace it to confirm the logic is correct:

**Starting:** facing E (index 4). Directions: `[SW=0, W=1, NW=2, NE=3, E=4, SE=5]`

| Step | From→To | Direction of move | Facing before | Rotations needed | Instructions |
|------|---------|-------------------|---------------|------------------|-------------|
| 1 | 24→25 | E (4) | E (4) | none | `→ Forward` |
| 2 | 25→18 | NW (2) | E (4) | 2 left | `↺ Rot L, ↺ Rot L, → Forward` |
| 3 | 18→11 | NE (3) | NW (2) | 1 right | `↻ Rot R, → Forward` |

Total: `[5, 6, 6, 5, 7, 5]` ✓

One subtlety worth noting: hex 18 is NE *of center*, but NW *of hex 25*. The converter correctly computes the step-by-step deltas rather than directions from origin, which is exactly what you want for piece movement.

The math is solid. Want to wire up the `/piece_editor_demo` route so you can test the full drag → POST → badge display loop in the browser?

Yes  wire up the /piece_editor_demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    return Titled("Piece Editor Demo",
        Div(
            H3("Rook Rule Editor"),
            P("Drag across hexes to create movement rules. "
              "The amber R is your rook, facing East.",
              cls="text-sm opacity-70 mb-4"),
            piece_drag_editor(dummy_rook),
            Divider(),
            H4("Facing Selector"),
            facing_grid(dummy_rook.id, current_facing=dummy_rook.facing),
            cls="p-6 space-y-6 max-w-xl mx-auto"
        )
    )
```

Navigate to `/piece_editor_demo` in the browser. You should be able to:

1. **Drag across hexes** → yellow highlight follows your finger/cursor → on release, coloured instruction badges appear below the grid
2. **Tap a direction** on the compass rose → the green highlight moves (though note the facing selector and drag editor are independent right now — changing facing on the compass doesn't update the drag grid's arrow)

Each drag **appends** to the existing rules (because of the `merged = existing + new_rules` logic in `/piece_path`). If you'd rather **replace**, change that line to `merged = new_rules`.

In [ ]:
def rules_display(rules: list[int], piece_id: str):
    """Compact badge list for a rule sequence, always with correct swap id."""
    target_id = f"rules-{piece_id[:8]}"

    if not rules:
        return Div(
            P("No rules yet — drag a path above", cls="text-sm opacity-50"),
            id=target_id,
        )

    badges = []
    for i, r in enumerate(rules):
        label, color = _INSTR_LABELS.get(r, (str(r), "#888"))
        badges.append(
            Span(label,
                 style=f"background:{color}22; border:1px solid {color}; "
                        "border-radius:4px; padding:2px 6px; font-size:11px; "
                       f"color:{color}; margin:2px; display:inline-block;")
        )

    # Count summary
    forwards = sum(1 for r in rules if r == 5)
    rotations = sum(1 for r in rules if r in (6, 7))
    actions = sum(1 for r in rules if r in (0, 1, 2, 3, 4))

    summary = f"{forwards} moves, {rotations} turns"
    if actions:
        summary += f", {actions} actions"

    return Div(
        P(f"{len(rules)} instructions — {summary}", cls="text-xs opacity-50 mb-1"),
        Div(*badges, style="line-height:2"),
        Pre(",".join(str(r) for r in rules),
            cls="text-xs opacity-40 mt-2 font-mono"),
        id=target_id,
    )


In [ ]:
@rt("/piece_editor_demo")
def piece_editor_demo(session):
    """Live demo page for the drag-to-rules editor."""
    try:
        logging.info("piece_editor_demo: building page")
        editor = piece_drag_editor(dummy_rook)
        logging.info("piece_editor_demo: editor built OK")
        facing = facing_grid(dummy_rook.id, current_facing=dummy_rook.facing)
        logging.info("piece_editor_demo: facing built OK")
        return Titled("Piece Editor Demo",
            Div(
                H3("Rook Rule Editor"),
                P("Drag across hexes to create movement rules.",
                  cls="text-sm opacity-70 mb-4"),
                editor,
                Divider(),
                H4("Facing Selector"),
                facing,
                cls="p-6 space-y-6 max-w-xl mx-auto"
            )
        )
    except Exception as e:
        logging.error(f"piece_editor_demo FAILED: {e}", exc_info=True)
        return P(f"Error: {e}")


@rt("/piece_path/{piece_id}/{facing:int}")
async def piece_path(request, piece_id: str, facing: int):
    """Receive dragged hex_ids, convert to rules, return badge display."""
    try:
        form = await request.form()
        raw = form.get("hex_ids", "")
        logging.info(f"piece_path: piece_id={piece_id} facing={facing} raw='{raw}'")

        if not raw:
            return Div(P("No path received", cls="text-sm opacity-50"),
                       id=f"rules-{piece_id[:8]}")

        hex_ids = [int(x) for x in raw.split(",") if x.strip()]
        logging.info(f"piece_path: {len(hex_ids)} hex_ids: {hex_ids}")

        # Reconstruct same grid as piece_drag_editor
        rings  = 3
        radius = 35
        dummy_style = StyleCSS("x", fill="white")
        grid = HexGrid.centered(rings, radius=radius, style=dummy_style)

        new_rules = path_to_rules(hex_ids, grid, start_facing=facing)
        logging.info(f"piece_path: {len(new_rules)} rules: {new_rules}")

        # For demo pieces, skip DB persistence
        if not piece_id.startswith("demo-"):
            try:
                rec = globalStore.pieces[piece_id]
                if rec:
                    existing_str = rec.rules if hasattr(rec, 'rules') else rec.get("rules", "")
                    existing = [int(x) for x in (existing_str or "").split(",") if x]
                    new_rules = existing + new_rules
                    globalStore.pieces.upsert(
                        {"id": piece_id, "rules": ",".join(str(r) for r in new_rules)},
                        pk="id"
                    )
            except Exception as db_err:
                logging.warning(f"piece_path: DB skip: {db_err}")

        return rules_display(new_rules, piece_id)

    except Exception as e:
        logging.error(f"piece_path FAILED: {e}", exc_info=True)
        return Div(P(f"Error: {e}", cls="text-sm text-red-500"),
                   id=f"rules-{piece_id[:8]}")


In [ ]:
!tail -10 base.text

This works and I am so excited. I want to do a commit which means I will loose the prompt history. can you write up where we have been and where we are going so I can put it into a note to come back to.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's your recap note:

```markdown
## Piece Editor — Progress & Roadmap

### What we built today

**1. Facing Selector (`facing_grid`)**
- 1-ring `HexGrid.centered` compass rose with `HexTouchMap`
- Center hex shows the piece, 6 neighbors show direction arrows (↙←↖↗→↘)
- Tapping a direction hex POSTs to `/set_facing/{piece_id}/{facing}` and swaps the widget
- Current facing highlighted in green (`fc-active` style)
- Direction indices match `HexPosition.directions()`: [SW=0, W=1, NW=2, NE=3, E=4, SE=5]

**2. Drag-to-Rules Editor (`piece_drag_editor`)**
- 3-ring `HexGrid.centered` with `HexDragMap` for drag selection
- Amber piece hex at center with facing arrow overlay
- Drag across hexes → JS collects `hex_ids` → POSTs to `/piece_path/{piece_id}/{facing}`
- Server reconstructs same grid, converts indices to `path_to_rules()`

**3. `path_to_rules()` converter**
- Takes list of hex grid indices + starting facing
- Deduplicates consecutive repeats from drag
- For each step: computes delta HexPosition, finds target direction index
- Generates shortest rotation (ROT_L/ROT_R) then FORWARD
- Verified: center→E→NE→NE with facing=E produces `[5, 6, 6, 5, 7, 5]` ✓

**4. `rules_display()` — badge visualization**
- Colour-coded instruction badges (blue=forward, grey=rotate, red=defend, green=harvest, etc.)
- Summary line: "6 instructions — 3 moves, 3 turns"
- Raw rule string shown in mono for debugging

**5. Routes**
- `GET /piece_editor_demo` — standalone test page with drag editor + facing selector
- `POST /piece_path/{piece_id}/{facing}` — receives drag, returns rules badges
- `GET /facing_selector/{piece_id}` — renders compass for a DB piece
- `POST /set_facing/{piece_id}/{facing}` — updates facing, returns refreshed compass
- Demo pieces (id starts with "demo-") skip DB persistence

**6. Instruction enum (in data.py)**
```
0=PAUSE, 1=DEFEND, 2=HARVEST, 3=GIVE, 4=SETTLE, 5=FORWARD, 6=ROT_L, 7=ROT_R
```

**7. Piece dataclass updates (already in data.py)**
- `rules: List[int]` replaces old string rule
- `cursor: int` — position in rules list
- `patrol: bool` — loop rules when cursor hits end
- `food/diet/food_capacity` — food economy fields
- `facing: int` — 0–5 direction index
- Full encode/decode for tab-delimited persistence

### What's next

**Immediate (rules editor polish):**
- Wire facing selector changes to update the drag editor's arrow
- Add "Clear rules" button alongside drag append
- Add manual instruction buttons (tap to append a single PAUSE/DEFEND/HARVEST/etc.)
- Support insert-at-cursor editing (not just append)

**Turn simulation:**
- `simulate_piece(piece, board, n_turns) -> list[SimStep]` — dry-run rules forward
- Each turn: execute instructions consuming `movement_range`, rotations free
- Non-movement actions (0–3) cost 1 point AND end the turn
- Forward blocked by water → rotate left once, stop (no spin trap)
- Overlay visualization: arrows for movement, circles for pause actions

**Turn execution model (settled design):**
- Rotations are free
- Instructions 0–3 cost 1 movement point and end the turn
- FORWARD costs `movement_cost()` (terrain-dependent, min 1)
- End-of-list: if last instruction is non-movement → idle; if movement → loop to start
- Priority: pieces resolve in chess rank order (King first, then Queen, etc.)

**Food economy:**
- Need `hex_food_yield(terrain, idx) -> float` based on climate/flow
- HARVEST adds `harvest_strength * yield` to `food` (capped at `food_capacity`)
- Each turn: `food -= diet`; if food < 0, lose health
- GIVE: transfer food to nearest adjacent friendly piece/settlement

**Integration with settlement view:**
- Clicking a citizen in settlement_left → loads piece_detail with facing + drag editor
- piece_detail goes in right-panel (already have the panel shell)
- Rules saved to DB via `piece.save()`

**Later:**
- Common rule presets (patrol circle, harvest-and-return, flanking)
- Dijkstra destination picker: tap a hex → auto-generate rules to reach it
- Gosper curve patrol generation anchored to settlements
- Collision resolution when two pieces move to same hex
- Settle/siege mechanics for instruction 4
```

That should get you right back in context after the commit. 🎉

## Piece Editor — Progress & Roadmap

### What we built today

**1. Facing Selector (`facing_grid`)**
- 1-ring `HexGrid.centered` compass rose with `HexTouchMap`
- Center hex shows the piece, 6 neighbors show direction arrows (↙←↖↗→↘)
- Tapping a direction hex POSTs to `/set_facing/{piece_id}/{facing}` and swaps the widget
- Current facing highlighted in green (`fc-active` style)
- Direction indices match `HexPosition.directions()`: [SW=0, W=1, NW=2, NE=3, E=4, SE=5]

**2. Drag-to-Rules Editor (`piece_drag_editor`)**
- 3-ring `HexGrid.centered` with `HexDragMap` for drag selection
- Amber piece hex at center with facing arrow overlay
- Drag across hexes → JS collects `hex_ids` → POSTs to `/piece_path/{piece_id}/{facing}`
- Server reconstructs same grid, converts indices to `path_to_rules()`

**3. `path_to_rules()` converter**
- Takes list of hex grid indices + starting facing
- Deduplicates consecutive repeats from drag
- For each step: computes delta HexPosition, finds target direction index
- Generates shortest rotation (ROT_L/ROT_R) then FORWARD
- Verified: center→E→NE→NE with facing=E produces `[5, 6, 6, 5, 7, 5]` ✓

**4. `rules_display()` — badge visualization**
- Colour-coded instruction badges (blue=forward, grey=rotate, red=defend, green=harvest, etc.)
- Summary line: "6 instructions — 3 moves, 3 turns"
- Raw rule string shown in mono for debugging

**5. Routes**
- `GET /piece_editor_demo` — standalone test page with drag editor + facing selector
- `POST /piece_path/{piece_id}/{facing}` — receives drag, returns rules badges
- `GET /facing_selector/{piece_id}` — renders compass for a DB piece
- `POST /set_facing/{piece_id}/{facing}` — updates facing, returns refreshed compass
- Demo pieces (id starts with "demo-") skip DB persistence

**6. Instruction enum (in data.py)**
